# Understanding AI Memory: From Legacy LangChain to LangGraph

<div style="background:linear-gradient(120deg,#0b1f3a,#123b58);color:white;padding:28px;border-radius:16px"><h2>Memory is a system design choice</h2><p style="color:#bfe9e4">What an agent remembers, how it recalls it, and how long it persists are separate questions.</p></div>

<style>h1,h2,h3{color:#123b58}blockquote{border-left:4px solid #19a89a;background:#eef8f7;padding:12px} table{border-collapse:collapse} th{background:#123b58;color:white} td,th{padding:8px;border:1px solid #d5e1e8} code{color:#7952a3}</style>


## Learning objectives

- Distinguish prompt history from application memory and persistence.
- Explain common legacy LangChain memory patterns and their limitations.
- Describe LangGraph thread state, checkpointers, and cross-thread stores.
- Choose an appropriate retrieval and context-management strategy.

> **Core model:** capture → store → retrieve → use → update or delete. Every stage needs an explicit policy.

## A useful mental model

Memory is not a single magical component. It is a pipeline connecting a signal (a user statement), a representation (messages, profile fields, or documents), a persistence layer, and a retrieval policy. The model only sees the context your application supplies.

```mermaid
flowchart LR
  E[Experience] --> X[Extract] --> S[Store] --> R[Retrieve relevant items] --> C[Prompt / state context] --> A[Agent response]
  A --> E
```

A durable database may hold facts that are never retrieved; a summary may be retrieved but omit a critical constraint. Evaluate the whole pipeline.

# Legacy LangChain Memory Patterns

These are older abstractions widely used by previous LangChain tutorials. They offered convenient prompt wiring, especially for chains, but coupled memory behavior to chain interfaces and often obscured storage, token costs, privacy, and retrieval. Many moved to `langchain-classic` in the 1.x ecosystem. The snippets below are **historical / illustrative** and are not required to run.

## A. ConversationBufferMemory

**Stores:** the full sequence of exchanged messages, commonly formatted into a prompt variable before each call.

**Advantages:** simple, transparent, preserves wording and chronology; useful for a short support exchange where exact recent wording matters.

**Costs:** the transcript grows with every turn, consumes context tokens, may expose unnecessary personal details, and eventually exceeds model limits.

```python
# HISTORICAL / illustrative (langchain-classic; API may vary by release)
from langchain_classic.memory import ConversationBufferMemory
memory = ConversationBufferMemory(return_messages=True)
memory.save_context({"input": "I prefer aisle seats"}, {"output": "Noted."})
history = memory.load_memory_variables({})["history"]
```

Modern alternative: keep messages in graph state with a checkpointer, and trim or summarize the prompt context when needed.

## B. ConversationBufferWindowMemory

A window keeps only a recent suffix, often expressed as the last *k* turns. Compared with a full buffer, token use is bounded approximately by recent conversation length rather than total history. A small `k` is cheap but can lose an older constraint (for example, an allergy or the requested travel dates); a large `k` approaches the full buffer.

```python
# HISTORICAL / illustrative
from langchain_classic.memory import ConversationBufferWindowMemory
memory = ConversationBufferWindowMemory(k=4, return_messages=True)
```

Use a window for local continuity, not as the sole store for enduring preferences. Preserve durable facts separately.

## C. ConversationSummaryMemory

An LLM periodically compresses earlier turns into a natural-language summary. The summary saves context compared with verbatim history and retains broad themes, but it is lossy: exact numbers, exceptions, chronology, or uncertainty can be dropped or distorted. Summaries also cost model calls and can accumulate errors.

```python
# HISTORICAL / illustrative; requires an LLM and classic compatibility package
from langchain_classic.memory import ConversationSummaryMemory
memory = ConversationSummaryMemory(llm=llm)
```

Keep source records or structured fields for details where exactness matters.

## D. Token-based memory / trimming

A token budget prunes messages until the retained context fits a model-specific token threshold. This differs from a fixed message window: message lengths vary, so “last 5 messages” can be tiny or enormous. Token trimming controls prompt size, but the oldest removed detail may be important.

```python
# Illustrative current context-management idea
from langchain_core.messages import trim_messages
short_context = trim_messages(messages, max_tokens=1200, token_counter=model, strategy="last")
```

Use token-aware trimming as a prompt-budget policy alongside persistent storage, not as durable memory.

## E. Summary-based memory

A running summary is updated as interactions occur; a periodic summary is created at checkpoints or when the transcript becomes long. It represents earlier conversation semantically, rather than preserving every utterance. It is appropriate for broad project context or a long-running conversation where exact quotes are unnecessary.

```python
# Pseudocode: application-owned summary policy
summary = summarize(previous_summary, newly_completed_messages)
store(thread_id, summary)
```

Compare summary against structured facts before relying on dates, budgets, names, or explicit user corrections.

## F. Vector / retriever-based memory

An embedding maps text to a numeric vector. A vector index can retrieve semantically similar fragments even when wording differs. A memory record can include text, metadata (user, date, confidence), and an identifier. At query time, retrieve a small relevant set and place it in context.

```python
# Illustrative; exact vector-store setup depends on provider
retriever = vector_store.as_retriever(search_kwargs={"k": 4})
memories = retriever.invoke("What food should I suggest in Florence?")
```

**Semantic memory** describes stable facts/knowledge (e.g., “prefers Italian food”). **Semantic search** is one retrieval technique. Structured lookup is often safer for exact budgets; vector retrieval helps discover relevant free-text episodes. Risks include irrelevant matches, stale facts, privacy leakage across users, and embedding/index cost.

## Other patterns and the key distinction

Entity memory tracks entities and relationships; knowledge graphs represent connected facts; explicit user profiles store typed fields; episodic memory stores events. These are representations or memory categories, not interchangeable persistence engines. A retriever can search many representations.

# Modern Memory in the LangGraph Ecosystem

**Short-term memory** is conversation state scoped to a thread. **Long-term memory** is information retrieved across threads or sessions, commonly scoped by a user namespace. They solve different recall problems.

## A. Short-term conversation memory

A graph state may contain messages and task data. A `thread_id` identifies the ongoing conversation; a checkpointer saves graph state at execution boundaries and restores it on later invocations using that same thread identifier. Different thread IDs do not automatically share state.

```python
from langgraph.checkpoint.memory import InMemorySaver
from langchain.agents import create_agent

agent = create_agent(model, tools, checkpointer=InMemorySaver())
config = {"configurable": {"thread_id": "trip-42"}}
agent.invoke({"messages": [{"role": "user", "content": "I like museums"}]}, config)
agent.invoke({"messages": [{"role": "user", "content": "What did I say I like?"}]}, config)
```

`InMemorySaver` is useful in a process-local example and disappears when the process exits. Durable checkpointers use a persistent backend and require backend-specific setup/lifecycle management. A checkpoint preserves a thread; it is not a cross-thread user profile.

## B. Persistent long-term memory

A store uses namespaces and keys to isolate durable records. Store the user ID in the namespace so another user's profile cannot be retrieved accidentally. An in-memory store is lost on restart; durable stores require a persistent backend.

```python
from langgraph.store.memory import InMemoryStore
store = InMemoryStore()  # process-local only
namespace = ("users", "user-123", "preferences")
store.put(namespace, "food", {"cuisine": "Italian"})
food = store.get(namespace, "food")
```

For durable preferences, use SQLite or a supported persistent LangGraph store/checkpointer adapter and test reopening the process. This project demonstrates application-owned SQLite profiles.

## C. Semantic memory (facts)

Semantic memory contains relatively stable facts: enjoys museums, prefers Italian food, avoids early starts, or has a stated budget range. Store structured profiles when fields have clear meaning and need exact updates. Keep uncertainty separate from confirmed facts.

```python
profile = {"likes_museums": True, "cuisine_preferences": ["Italian"],
           "avoid_early_mornings": True, "budget_min": None, "budget_max": None}
```

Never fill an unspecified budget with a guess. A correction should update the relevant field without silently discarding unrelated preferences.

## D. Episodic memory (events)

Episodes capture what happened: a Florence itinerary was proposed, a hotel was selected, or the user rejected a museum-heavy plan. Store a compact event with date, request, outcome, and user feedback. Retrieve only episodes relevant to the current task.

```python
episode = {"event": "planned_trip", "destination": "Florence",
           "outcome": "user selected central hotel", "feedback": "keep mornings relaxed"}
```

Episodes complement stable profile facts; an old choice should not automatically become a permanent preference.

## E. Procedural memory (behavior)

Procedural memory guides the agent's actions: retrieve preferences before planning, show a plan before tools, validate dependencies, and verify tool results before writing the itinerary. It can live in code, prompts, policies, or stored procedures when users need customization. Not every rule belongs in a database.

```python
POLICY = ["load user profile", "show validated plan",
          "execute dependent tools", "use returned data in final answer"]
```

## F. Extraction, retrieval, and stewardship

- **Extraction:** prefer explicit user statements; attach confidence and provenance to inferences.
- **Update/consolidation:** merge duplicate facts; resolve conflicts using recency and explicit correction rules.
- **Retrieval:** structured lookup for exact fields; semantic search for fuzzy episodes; hybrid retrieval when needed.
- **Privacy:** collect only useful data, isolate by user, support inspection and deletion, and set retention limits.
- **Timing:** update important preferences during the interaction; defer expensive consolidation to background work if appropriate.

Memory quality depends on what is saved and whether the relevant item is recalled at the right time.

## Comparison and migration

| Approach | Representation | Recall scope | Persistence | Context impact | Retrieval | Best fit / limitation |
|---|---|---|---|---|---|---|
| Buffer | Verbatim messages | Whole chain | Often process/chain configured | Grows quickly | Append all | Exact short exchanges; token growth |
| Window | Recent messages | Last *k* turns | Same caveat | Bounded by turns | Recent suffix | Local continuity; old constraints disappear |
| Summary | Compressed prose | Earlier themes | Depends on backend | Compact | Insert summary | Long context; lossy and may drift |
| Token trim | Selected messages | Budgeted suffix | Usually separate | Explicit token cap | Prune by count | Prompt control; not durable memory |
| Retriever | Documents/fragments | Relevant matches | Vector-store dependent | Small retrieved subset | Similarity/filters | Fuzzy recall; relevance and isolation matter |
| Checkpointer | Graph/thread state | One thread | In-memory or durable backend | State is selectively sent to model | `thread_id` | Resume interaction; not cross-thread profile |
| Long-term store | Key/value, records, documents | User/app namespace | Backend dependent | Only retrieved items | Key, filters, semantic search | Cross-thread facts; needs lifecycle/privacy policy |

These are not direct equivalents: checkpointing persists thread execution state; a long-term store shares selected records across threads; a vector retriever is a retrieval technique; summarization and trimming manage model context.

## Migration guide

1. Inventory what each old memory object actually stores and who may read it.
2. Separate thread history from durable user facts and past events.
3. Put exact preferences in a typed profile; keep uncertainty/provenance explicit.
4. Use a thread checkpointer for resumable state and a user-scoped store for cross-thread facts.
5. Add retrieval filters and token trimming/summarization for context management.
6. Test restart persistence, user isolation, corrections, deletion, and missing-data behavior.
7. Compare responses before and after migration; memory presence alone does not prove correct recall.

## Key takeaways

1. The model only “remembers” what the application retrieves into context.
2. Thread state and cross-session user memory have different scopes.
3. Summaries and token trimming are context strategies, not substitutes for durable records.
4. Typed facts are safer for exact preferences; retrieval is useful for relevant episodes.
5. Privacy, user isolation, corrections, and deletion are core memory features.

### Discussion questions

- Which trip details should be structured facts versus episodes?
- When should an inferred preference be confirmed?
- How would you resolve a new preference that conflicts with an old one?
- What should be deleted when a user asks to forget a trip?

### Official references

- [LangChain agents](https://docs.langchain.com/oss/python/langchain/agents)
- [LangGraph persistence](https://docs.langchain.com/oss/python/langgraph/persistence)
- [LangGraph memory](https://docs.langchain.com/oss/python/langgraph/add-memory)
- [LangChain classic package](https://reference.langchain.com/python/langchain-classic/)